In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [ ]:
name_dataset = 'SSTv2'

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post21/df_test'

# 2. Load Environment

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import numpy as np
import pandas as pd

# 3. Load Datasets

In [8]:
n_warmup = 10

In [9]:
all_df = []

In [10]:
for i in range(1, 5 + 1):
  path_df_part = f'{path_open}_{i}.csv'
  df_part = pd.read_csv(path_df_part)

  N_part = len(df_part)
  warm_up_col_first = ['True' for i in range(n_warmup)]
  warm_up_col_second = ['False' for i in range(N_part - n_warmup)]
  warm_up_col = warm_up_col_first + warm_up_col_second
  df_part['warmup'] = warm_up_col

  all_df.append(df_part)

In [11]:
df = pd.concat(all_df, axis = 0)

In [12]:
df = df.reset_index()
df = df.drop(columns = ['index', 'Unnamed: 0'])

In [13]:
df.shape

(3800, 81)

# 4. Analysis

In [14]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

### t-student CI

$ X_1, X_2, X_3 $

$ \bar{x} = \frac{1}{3} \cdot \sum_{i=1}^3 x_i $

$ s = \sqrt{\frac{\sum_{i=1}^{3} (x_i - \bar{x})^2}{3-1}} $

$ SE = \cfrac{s}{\sqrt{3}} $

$ CI = \bar{x} \pm t_{0.975,2} \cdot SE $

$ t_{0.975,2} \approx 4.303 $

### Latency

Latency 50 = median(Latency 50 - seed 1, Latency 50 - seed 2, Latency 50 - seed 3)

Latency 95 = median(Latency 95 - seed 1, Latency 95 - seed 2, Latency 95 - seed 3)

Latency 99 = median(Latency 99 - seed 1, Latency 99 - seed 2, Latency 99 - seed 3)

In [15]:
def confidence_interval(x_1, x_2, x_3):

  bar_x = (x_1 + x_2 + x_3)/3

  squared_s = ((bar_x - x_1)**2 + (bar_x - x_2)**2 + (bar_x - x_3)**2)/2

  s = (squared_s)**(1/2)

  SE = s/(3**(1/2))

  tail = 4.303*SE

  return round(bar_x, 2), round(tail, 2)

In [16]:
f_warmup = (df['warmup'] == 'False')

### a. BERT

In [17]:
model = 'bert-base-uncased'

**i. F1**

In [18]:
f1_1 = f1_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
f1_2 = f1_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
f1_3 = f1_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_f1, tail_f1 = confidence_interval(f1_1, f1_2, f1_3)
print(bar_f1, tail_f1)
print(str(bar_f1) + '±' + str(tail_f1))

94.43 0.24
94.43±0.24


**ii. Precision**

In [19]:
prec_1 = precision_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
prec_2 = precision_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
prec_3 = precision_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_prec, tail_prec = confidence_interval(prec_1, prec_2, prec_3)
print(bar_prec, tail_prec)
print(str(bar_prec) + '±' + str(tail_prec))

94.46 0.23
94.46±0.23


**iii. Recall**

In [20]:
rec_1 = recall_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
rec_2 = recall_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
rec_3 = recall_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_rec, tail_rec = confidence_interval(rec_1, rec_2, rec_3)
print(bar_rec, tail_rec)
print(str(bar_rec) + '±' + str(tail_rec))

94.42 0.24
94.42±0.24


**iv. Accuracy**

In [21]:
acc_1 = accuracy_score(df['label'], df[f'{model}-seed-1-label'])*100
acc_2 = accuracy_score(df['label'], df[f'{model}-seed-2-label'])*100
acc_3 = accuracy_score(df['label'], df[f'{model}-seed-3-label'])*100
bar_acc, tail_acc = confidence_interval(acc_1, acc_2, acc_3)
print(bar_acc, tail_acc)
print(str(bar_acc) + '±' + str(tail_acc))

94.42 0.24
94.42±0.24


**v. Latency 50, 95, 99**

In [22]:
l_50_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.50)
l_95_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.95)
l_99_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.99)

l_50_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.50)
l_95_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.95)
l_99_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.99)

l_50_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.50)
l_95_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.95)
l_99_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.99)

In [23]:
l_50 = round(float(np.median([l_50_1, l_50_2, l_50_3])), 2)

In [24]:
l_50

196.91

In [25]:
l_95 = round(float(np.median([l_95_1, l_95_2, l_95_3])), 2)

In [26]:
l_95

285.8

In [27]:
l_99 = round(float(np.median([l_99_1, l_99_2, l_99_3])), 2)

In [28]:
l_99

572.99

### b. DistilBERT

In [29]:
model = 'distilbert-base-uncased'

**i. F1**

In [30]:
f1_1 = f1_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
f1_2 = f1_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
f1_3 = f1_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_f1, tail_f1 = confidence_interval(f1_1, f1_2, f1_3)
print(bar_f1, tail_f1)
print(str(bar_f1) + '±' + str(tail_f1))

94.11 0.17
94.11±0.17


**ii. Precision**

In [31]:
prec_1 = precision_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
prec_2 = precision_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
prec_3 = precision_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_prec, tail_prec = confidence_interval(prec_1, prec_2, prec_3)
print(bar_prec, tail_prec)
print(str(bar_prec) + '±' + str(tail_prec))

94.14 0.22
94.14±0.22


**iii. Recall**

In [32]:
rec_1 = recall_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
rec_2 = recall_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
rec_3 = recall_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_rec, tail_rec = confidence_interval(rec_1, rec_2, rec_3)
print(bar_rec, tail_rec)
print(str(bar_rec) + '±' + str(tail_rec))

94.11 0.17
94.11±0.17


**iv. Accuracy**

In [33]:
acc_1 = accuracy_score(df['label'], df[f'{model}-seed-1-label'])*100
acc_2 = accuracy_score(df['label'], df[f'{model}-seed-2-label'])*100
acc_3 = accuracy_score(df['label'], df[f'{model}-seed-3-label'])*100
bar_acc, tail_acc = confidence_interval(acc_1, acc_2, acc_3)
print(bar_acc, tail_acc)
print(str(bar_acc) + '±' + str(tail_acc))

94.11 0.17
94.11±0.17


**v. Latency 50, 95, 99**

In [34]:
l_50_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.50)
l_95_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.95)
l_99_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.99)

l_50_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.50)
l_95_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.95)
l_99_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.99)

l_50_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.50)
l_95_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.95)
l_99_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.99)

In [35]:
l_50 = round(float(np.median([l_50_1, l_50_2, l_50_3])), 2)

In [36]:
l_50

108.19

In [37]:
l_95 = round(float(np.median([l_95_1, l_95_2, l_95_3])), 2)

In [38]:
l_95

161.32

In [39]:
l_99 = round(float(np.median([l_99_1, l_99_2, l_99_3])), 2)

In [40]:
l_99

515.73

### c. RoBERTa

In [41]:
model = 'roberta-base'

**i. F1**

In [42]:
f1_1 = f1_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
f1_2 = f1_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
f1_3 = f1_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_f1, tail_f1 = confidence_interval(f1_1, f1_2, f1_3)
print(bar_f1, tail_f1)
print(str(bar_f1) + '±' + str(tail_f1))

94.63 0.35
94.63±0.35


**ii. Precision**

In [43]:
prec_1 = precision_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
prec_2 = precision_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
prec_3 = precision_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_prec, tail_prec = confidence_interval(prec_1, prec_2, prec_3)
print(bar_prec, tail_prec)
print(str(bar_prec) + '±' + str(tail_prec))

94.65 0.33
94.65±0.33


**iii. Recall**

In [44]:
rec_1 = recall_score(df['label'], df[f'{model}-seed-1-label'], average = 'macro')*100
rec_2 = recall_score(df['label'], df[f'{model}-seed-2-label'], average = 'macro')*100
rec_3 = recall_score(df['label'], df[f'{model}-seed-3-label'], average = 'macro')*100
bar_rec, tail_rec = confidence_interval(rec_1, rec_2, rec_3)
print(bar_rec, tail_rec)
print(str(bar_rec) + '±' + str(tail_rec))

94.62 0.36
94.62±0.36


**iv. Accuracy**

In [45]:
acc_1 = accuracy_score(df['label'], df[f'{model}-seed-1-label'])*100
acc_2 = accuracy_score(df['label'], df[f'{model}-seed-2-label'])*100
acc_3 = accuracy_score(df['label'], df[f'{model}-seed-3-label'])*100
bar_acc, tail_acc = confidence_interval(acc_1, acc_2, acc_3)
print(bar_acc, tail_acc)
print(str(bar_acc) + '±' + str(tail_acc))

94.62 0.36
94.62±0.36


**v. Latency 50, 95, 99**

In [46]:
l_50_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.50)
l_95_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.95)
l_99_1 = df[f_warmup][f'{model}-seed-1-latency'].quantile(q = 0.99)

l_50_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.50)
l_95_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.95)
l_99_2 = df[f_warmup][f'{model}-seed-2-latency'].quantile(q = 0.99)

l_50_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.50)
l_95_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.95)
l_99_3 = df[f_warmup][f'{model}-seed-3-latency'].quantile(q = 0.99)

In [47]:
l_50 = round(float(np.median([l_50_1, l_50_2, l_50_3])), 2)

In [48]:
l_50

188.7

In [49]:
l_95 = round(float(np.median([l_95_1, l_95_2, l_95_3])), 2)

In [50]:
l_95

279.2

In [51]:
l_99 = round(float(np.median([l_99_1, l_99_2, l_99_3])), 2)

In [52]:
l_99

485.33

### d. GPT 4o - Zero Shot

In [53]:
model = 'gpt-4o-zero'

In [54]:
f_p_1 = (df[f'{model}-seed-1-label'] != -1)
f_p_2 = (df[f'{model}-seed-2-label'] != -1)
f_p_3 = (df[f'{model}-seed-3-label'] != -1)

**i. F1**

In [55]:
f1_1 = f1_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
f1_2 = f1_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
f1_3 = f1_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_f1, tail_f1 = confidence_interval(f1_1, f1_2, f1_3)
print(bar_f1, tail_f1)
print(str(bar_f1) + '±' + str(tail_f1))

87.93 0.63
87.93±0.63


**ii. Precision**

In [56]:
prec_1 = precision_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
prec_2 = precision_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
prec_3 = precision_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_prec, tail_prec = confidence_interval(prec_1, prec_2, prec_3)
print(bar_prec, tail_prec)
print(str(bar_prec) + '±' + str(tail_prec))

88.22 0.6
88.22±0.6


**iii. Recall**

In [57]:
rec_1 = recall_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
rec_2 = recall_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
rec_3 = recall_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_rec, tail_rec = confidence_interval(rec_1, rec_2, rec_3)
print(bar_rec, tail_rec)
print(str(bar_rec) + '±' + str(tail_rec))

88.0 0.62
88.0±0.62


**iv. Accuracy**

In [58]:
acc_1 = accuracy_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'])*100
acc_2 = accuracy_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'])*100
acc_3 = accuracy_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'])*100
bar_acc, tail_acc = confidence_interval(acc_1, acc_2, acc_3)
print(bar_acc, tail_acc)
print(str(bar_acc) + '±' + str(tail_acc))

88.0 0.62
88.0±0.62


**v. Latency 50, 95, 99**

In [59]:
f_l_1 = (df[f'{model}-seed-1-latency'] != '-')
f_l_2 = (df[f'{model}-seed-2-latency'] != '-')
f_l_3 = (df[f'{model}-seed-3-latency'] != '-')

In [60]:
l_50_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.50)
l_95_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.95)
l_99_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.99)

l_50_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.50)
l_95_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.95)
l_99_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.99)

l_50_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.50)
l_95_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.95)
l_99_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.99)

In [61]:
l_50 = round(float(np.median([l_50_1, l_50_2, l_50_3])), 2)

In [62]:
l_50

410.81

In [63]:
l_95 = round(float(np.median([l_95_1, l_95_2, l_95_3])), 2)

In [64]:
l_95

693.55

In [65]:
l_99 = round(float(np.median([l_99_1, l_99_2, l_99_3])), 2)

In [66]:
l_99

1249.39

**vi. TTFT 50, 95, 99**

In [67]:
f_t_1 = (df[f'{model}-seed-1-ttft'] != '-')
f_t_2 = (df[f'{model}-seed-2-ttft'] != '-')
f_t_3 = (df[f'{model}-seed-3-ttft'] != '-')

In [68]:
t_50_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.50)
t_95_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.95)
t_99_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.99)

t_50_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.50)
t_95_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.95)
t_99_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.99)

t_50_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.50)
t_95_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.95)
t_99_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.99)

In [69]:
t_50 = round(float(np.median([t_50_1, t_50_2, t_50_3])), 2)

In [70]:
t_50

404.67

In [71]:
t_95 = round(float(np.median([t_95_1, t_95_2, t_95_3])), 2)

In [72]:
t_95

686.61

In [73]:
t_99 = round(float(np.median([t_99_1, t_99_2, t_99_3])), 2)

In [74]:
t_99

1241.98

**vii. Average Input and Output Tokens**

In [75]:
f_it_1 = (df[f'{model}-seed-1-input-tokens'] != '-')
f_it_2 = (df[f'{model}-seed-2-input-tokens'] != '-')
f_it_3 = (df[f'{model}-seed-3-input-tokens'] != '-')

In [76]:
it_1 = (df[f_it_1][f'{model}-seed-1-input-tokens'].astype(float)).mean()
it_2 = (df[f_it_2][f'{model}-seed-2-input-tokens'].astype(float)).mean()
it_3 = (df[f_it_3][f'{model}-seed-3-input-tokens'].astype(float)).mean()
it = round(float((it_1 + it_2 + it_3)/3), 2)
print(it)

106.4


In [77]:
f_ot_1 = (df[f'{model}-seed-1-output-tokens'] != '-')
f_ot_2 = (df[f'{model}-seed-2-output-tokens'] != '-')
f_ot_3 = (df[f'{model}-seed-3-output-tokens'] != '-')

In [78]:
ot_1 = (df[f_ot_1][f'{model}-seed-1-output-tokens'].astype(float)).mean()
ot_2 = (df[f_ot_2][f'{model}-seed-2-output-tokens'].astype(float)).mean()
ot_3 = (df[f_ot_3][f'{model}-seed-3-output-tokens'].astype(float)).mean()
ot = round(float((ot_1 + ot_2 + ot_3)/3), 2)
print(ot)

1.0


### e. GPT 4o - Few Shot

In [79]:
model = 'gpt-4o-few'

In [80]:
f_p_1 = (df[f'{model}-seed-1-label'] != -1)
f_p_2 = (df[f'{model}-seed-2-label'] != -1)
f_p_3 = (df[f'{model}-seed-3-label'] != -1)

**i. F1**

In [81]:
f1_1 = f1_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
f1_2 = f1_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
f1_3 = f1_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_f1, tail_f1 = confidence_interval(f1_1, f1_2, f1_3)
print(bar_f1, tail_f1)
print(str(bar_f1) + '±' + str(tail_f1))

89.65 0.33
89.65±0.33


**ii. Precision**

In [82]:
prec_1 = precision_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
prec_2 = precision_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
prec_3 = precision_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_prec, tail_prec = confidence_interval(prec_1, prec_2, prec_3)
print(bar_prec, tail_prec)
print(str(bar_prec) + '±' + str(tail_prec))

89.73 0.34
89.73±0.34


**iii. Recall**

In [83]:
rec_1 = recall_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
rec_2 = recall_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
rec_3 = recall_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_rec, tail_rec = confidence_interval(rec_1, rec_2, rec_3)
print(bar_rec, tail_rec)
print(str(bar_rec) + '±' + str(tail_rec))

89.68 0.33
89.68±0.33


**iv. Accuracy**

In [84]:
acc_1 = accuracy_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'])*100
acc_2 = accuracy_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'])*100
acc_3 = accuracy_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'])*100
bar_acc, tail_acc = confidence_interval(acc_1, acc_2, acc_3)
print(bar_acc, tail_acc)
print(str(bar_acc) + '±' + str(tail_acc))

89.68 0.33
89.68±0.33


**v. Latency 50, 95, 99**

In [85]:
f_l_1 = (df[f'{model}-seed-1-latency'] != '-')
f_l_2 = (df[f'{model}-seed-2-latency'] != '-')
f_l_3 = (df[f'{model}-seed-3-latency'] != '-')

In [86]:
l_50_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.50)
l_95_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.95)
l_99_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.99)

l_50_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.50)
l_95_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.95)
l_99_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.99)

l_50_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.50)
l_95_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.95)
l_99_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.99)

In [87]:
l_50 = round(float(np.median([l_50_1, l_50_2, l_50_3])), 2)

In [88]:
l_50

332.31

In [89]:
l_95 = round(float(np.median([l_95_1, l_95_2, l_95_3])), 2)

In [90]:
l_95

582.24

In [91]:
l_99 = round(float(np.median([l_99_1, l_99_2, l_99_3])), 2)

In [92]:
l_99

955.58

**vi. TTFT 50, 95, 99**

In [93]:
f_t_1 = (df[f'{model}-seed-1-ttft'] != '-')
f_t_2 = (df[f'{model}-seed-2-ttft'] != '-')
f_t_3 = (df[f'{model}-seed-3-ttft'] != '-')

In [94]:
t_50_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.50)
t_95_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.95)
t_99_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.99)

t_50_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.50)
t_95_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.95)
t_99_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.99)

t_50_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.50)
t_95_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.95)
t_99_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.99)

In [95]:
t_50 = round(float(np.median([t_50_1, t_50_2, t_50_3])), 2)

In [96]:
t_50

328.7

In [97]:
t_95 = round(float(np.median([t_95_1, t_95_2, t_95_3])), 2)

In [98]:
t_95

578.81

In [99]:
t_99 = round(float(np.median([t_99_1, t_99_2, t_99_3])), 2)

In [100]:
t_99

954.27

**vii. Average Input and Output Tokens**

In [101]:
f_it_1 = (df[f'{model}-seed-1-input-tokens'] != '-')
f_it_2 = (df[f'{model}-seed-2-input-tokens'] != '-')
f_it_3 = (df[f'{model}-seed-3-input-tokens'] != '-')

In [102]:
it_1 = (df[f_it_1][f'{model}-seed-1-input-tokens'].astype(float)).mean()
it_2 = (df[f_it_2][f'{model}-seed-2-input-tokens'].astype(float)).mean()
it_3 = (df[f_it_3][f'{model}-seed-3-input-tokens'].astype(float)).mean()
it = round(float((it_1 + it_2 + it_3)/3), 2)
print(it)

357.4


In [103]:
f_ot_1 = (df[f'{model}-seed-1-output-tokens'] != '-')
f_ot_2 = (df[f'{model}-seed-2-output-tokens'] != '-')
f_ot_3 = (df[f'{model}-seed-3-output-tokens'] != '-')

In [104]:
ot_1 = (df[f_ot_1][f'{model}-seed-1-output-tokens'].astype(float)).mean()
ot_2 = (df[f_ot_2][f'{model}-seed-2-output-tokens'].astype(float)).mean()
ot_3 = (df[f_ot_3][f'{model}-seed-3-output-tokens'].astype(float)).mean()
ot = round(float((ot_1 + ot_2 + ot_3)/3), 2)
print(ot)

1.0


### f. Claude 4.5 - Zero Shot

In [105]:
model = 'claude-4.5-zero'

In [106]:
f_p_1 = (df[f'{model}-seed-1-label'] != -1)
f_p_2 = (df[f'{model}-seed-2-label'] != -1)
f_p_3 = (df[f'{model}-seed-3-label'] != -1)

**i. F1**

In [107]:
f1_1 = f1_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
f1_2 = f1_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
f1_3 = f1_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_f1, tail_f1 = confidence_interval(f1_1, f1_2, f1_3)
print(bar_f1, tail_f1)
print(str(bar_f1) + '±' + str(tail_f1))

91.35 0.28
91.35±0.28


**ii. Precision**

In [108]:
prec_1 = precision_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
prec_2 = precision_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
prec_3 = precision_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_prec, tail_prec = confidence_interval(prec_1, prec_2, prec_3)
print(bar_prec, tail_prec)
print(str(bar_prec) + '±' + str(tail_prec))

91.4 0.28
91.4±0.28


**iii. Recall**

In [109]:
rec_1 = recall_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
rec_2 = recall_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
rec_3 = recall_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_rec, tail_rec = confidence_interval(rec_1, rec_2, rec_3)
print(bar_rec, tail_rec)
print(str(bar_rec) + '±' + str(tail_rec))

91.36 0.27
91.36±0.27


**iv. Accuracy**

In [110]:
acc_1 = accuracy_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'])*100
acc_2 = accuracy_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'])*100
acc_3 = accuracy_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'])*100
bar_acc, tail_acc = confidence_interval(acc_1, acc_2, acc_3)
print(bar_acc, tail_acc)
print(str(bar_acc) + '±' + str(tail_acc))

91.36 0.27
91.36±0.27


**v. Latency 50, 95, 99**

In [111]:
f_l_1 = (df[f'{model}-seed-1-latency'] != '-')
f_l_2 = (df[f'{model}-seed-2-latency'] != '-')
f_l_3 = (df[f'{model}-seed-3-latency'] != '-')

In [112]:
l_50_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.50)
l_95_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.95)
l_99_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.99)

l_50_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.50)
l_95_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.95)
l_99_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.99)

l_50_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.50)
l_95_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.95)
l_99_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.99)

In [113]:
l_50 = round(float(np.median([l_50_1, l_50_2, l_50_3])), 2)

In [114]:
l_50

1434.82

In [115]:
l_95 = round(float(np.median([l_95_1, l_95_2, l_95_3])), 2)

In [116]:
l_95

2298.01

In [117]:
l_99 = round(float(np.median([l_99_1, l_99_2, l_99_3])), 2)

In [118]:
l_99

4551.21

**vi. TTFT 50, 95, 99**

In [119]:
f_t_1 = (df[f'{model}-seed-1-ttft'] != '-')
f_t_2 = (df[f'{model}-seed-2-ttft'] != '-')
f_t_3 = (df[f'{model}-seed-3-ttft'] != '-')

In [120]:
t_50_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.50)
t_95_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.95)
t_99_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.99)

t_50_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.50)
t_95_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.95)
t_99_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.99)

t_50_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.50)
t_95_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.95)
t_99_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.99)

In [121]:
t_50 = round(float(np.median([t_50_1, t_50_2, t_50_3])), 2)

In [122]:
t_50

1363.53

In [123]:
t_95 = round(float(np.median([t_95_1, t_95_2, t_95_3])), 2)

In [124]:
t_95

2212.85

In [125]:
t_99 = round(float(np.median([t_99_1, t_99_2, t_99_3])), 2)

In [126]:
t_99

4477.2

**vii. Average Input and Output Tokens**

In [127]:
f_it_1 = (df[f'{model}-seed-1-input-tokens'] != '-')
f_it_2 = (df[f'{model}-seed-2-input-tokens'] != '-')
f_it_3 = (df[f'{model}-seed-3-input-tokens'] != '-')

In [128]:
it_1 = (df[f_it_1][f'{model}-seed-1-input-tokens'].astype(float)).mean()
it_2 = (df[f_it_2][f'{model}-seed-2-input-tokens'].astype(float)).mean()
it_3 = (df[f_it_3][f'{model}-seed-3-input-tokens'].astype(float)).mean()
it = round(float((it_1 + it_2 + it_3)/3), 2)
print(it)

121.86


In [129]:
f_ot_1 = (df[f'{model}-seed-1-output-tokens'] != '-')
f_ot_2 = (df[f'{model}-seed-2-output-tokens'] != '-')
f_ot_3 = (df[f'{model}-seed-3-output-tokens'] != '-')

In [130]:
ot_1 = (df[f_ot_1][f'{model}-seed-1-output-tokens'].astype(float)).mean()
ot_2 = (df[f_ot_2][f'{model}-seed-2-output-tokens'].astype(float)).mean()
ot_3 = (df[f_ot_3][f'{model}-seed-3-output-tokens'].astype(float)).mean()
ot = round(float((ot_1 + ot_2 + ot_3)/3), 2)
print(ot)

5.0


### g. Claude 4.5 - Few Shot

In [131]:
model = 'claude-4.5-few'

In [132]:
f_p_1 = (df[f'{model}-seed-1-label'] != -1)
f_p_2 = (df[f'{model}-seed-2-label'] != -1)
f_p_3 = (df[f'{model}-seed-3-label'] != -1)

**i. F1**

In [133]:
f1_1 = f1_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
f1_2 = f1_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
f1_3 = f1_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_f1, tail_f1 = confidence_interval(f1_1, f1_2, f1_3)
print(bar_f1, tail_f1)
print(str(bar_f1) + '±' + str(tail_f1))

90.56 0.14
90.56±0.14


**ii. Precision**

In [134]:
prec_1 = precision_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
prec_2 = precision_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
prec_3 = precision_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_prec, tail_prec = confidence_interval(prec_1, prec_2, prec_3)
print(bar_prec, tail_prec)
print(str(bar_prec) + '±' + str(tail_prec))

90.6 0.15
90.6±0.15


**iii. Recall**

In [135]:
rec_1 = recall_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'], average = 'macro')*100
rec_2 = recall_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'], average = 'macro')*100
rec_3 = recall_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'], average = 'macro')*100
bar_rec, tail_rec = confidence_interval(rec_1, rec_2, rec_3)
print(bar_rec, tail_rec)
print(str(bar_rec) + '±' + str(tail_rec))

90.59 0.14
90.59±0.14


**iv. Accuracy**

In [136]:
acc_1 = accuracy_score(df[f_p_1]['label'], df[f_p_1][f'{model}-seed-1-label'])*100
acc_2 = accuracy_score(df[f_p_2]['label'], df[f_p_2][f'{model}-seed-2-label'])*100
acc_3 = accuracy_score(df[f_p_3]['label'], df[f_p_3][f'{model}-seed-3-label'])*100
bar_acc, tail_acc = confidence_interval(acc_1, acc_2, acc_3)
print(bar_acc, tail_acc)
print(str(bar_acc) + '±' + str(tail_acc))

90.59 0.14
90.59±0.14


**v. Latency 50, 95, 99**

In [137]:
f_l_1 = (df[f'{model}-seed-1-latency'] != '-')
f_l_2 = (df[f'{model}-seed-2-latency'] != '-')
f_l_3 = (df[f'{model}-seed-3-latency'] != '-')

In [138]:
l_50_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.50)
l_95_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.95)
l_99_1 = (df[f_warmup & f_l_1][f'{model}-seed-1-latency'].astype(float)).quantile(q = 0.99)

l_50_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.50)
l_95_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.95)
l_99_2 = (df[f_warmup & f_l_2][f'{model}-seed-2-latency'].astype(float)).quantile(q = 0.99)

l_50_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.50)
l_95_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.95)
l_99_3 = (df[f_warmup & f_l_3][f'{model}-seed-3-latency'].astype(float)).quantile(q = 0.99)

In [139]:
l_50 = round(float(np.median([l_50_1, l_50_2, l_50_3])), 2)

In [140]:
l_50

1002.56

In [141]:
l_95 = round(float(np.median([l_95_1, l_95_2, l_95_3])), 2)

In [142]:
l_95

2544.03

In [143]:
l_99 = round(float(np.median([l_99_1, l_99_2, l_99_3])), 2)

In [144]:
l_99

5756.15

**vi. TTFT 50, 95, 99**

In [145]:
f_t_1 = (df[f'{model}-seed-1-ttft'] != '-')
f_t_2 = (df[f'{model}-seed-2-ttft'] != '-')
f_t_3 = (df[f'{model}-seed-3-ttft'] != '-')

In [146]:
t_50_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.50)
t_95_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.95)
t_99_1 = (df[f_warmup & f_t_1][f'{model}-seed-1-ttft'].astype(float)).quantile(q = 0.99)

t_50_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.50)
t_95_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.95)
t_99_2 = (df[f_warmup & f_t_2][f'{model}-seed-2-ttft'].astype(float)).quantile(q = 0.99)

t_50_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.50)
t_95_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.95)
t_99_3 = (df[f_warmup & f_t_3][f'{model}-seed-3-ttft'].astype(float)).quantile(q = 0.99)

In [147]:
t_50 = round(float(np.median([t_50_1, t_50_2, t_50_3])), 2)

In [148]:
t_50

916.85

In [149]:
t_95 = round(float(np.median([t_95_1, t_95_2, t_95_3])), 2)

In [150]:
t_95

2441.21

In [151]:
t_99 = round(float(np.median([t_99_1, t_99_2, t_99_3])), 2)

In [152]:
t_99

5719.61

**vii. Average Input and Output Tokens**

In [153]:
f_it_1 = (df[f'{model}-seed-1-input-tokens'] != '-')
f_it_2 = (df[f'{model}-seed-2-input-tokens'] != '-')
f_it_3 = (df[f'{model}-seed-3-input-tokens'] != '-')

In [154]:
it_1 = (df[f_it_1][f'{model}-seed-1-input-tokens'].astype(float)).mean()
it_2 = (df[f_it_2][f'{model}-seed-2-input-tokens'].astype(float)).mean()
it_3 = (df[f_it_3][f'{model}-seed-3-input-tokens'].astype(float)).mean()
it = round(float((it_1 + it_2 + it_3)/3), 2)
print(it)

398.86


In [155]:
f_ot_1 = (df[f'{model}-seed-1-output-tokens'] != '-')
f_ot_2 = (df[f'{model}-seed-2-output-tokens'] != '-')
f_ot_3 = (df[f'{model}-seed-3-output-tokens'] != '-')

In [156]:
ot_1 = (df[f_ot_1][f'{model}-seed-1-output-tokens'].astype(float)).mean()
ot_2 = (df[f_ot_2][f'{model}-seed-2-output-tokens'].astype(float)).mean()
ot_3 = (df[f_ot_3][f'{model}-seed-3-output-tokens'].astype(float)).mean()
ot = round(float((ot_1 + ot_2 + ot_3)/3), 2)
print(ot)

5.0


# 5. Execution Time

In [157]:
end_notebook = time.time()

In [158]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 0m 11.66s
